In [40]:
import string
import io
import nltk
from gensim.models import Word2Vec
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# Descarregar recursos necessários do NLTK
nltk.download('punkt')
nltk.download('stopwords')

print("Recursos carregados com sucesso!")

Recursos carregados com sucesso!


[nltk_data] Downloading package punkt to /Users/gugafm11/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gugafm11/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Nesta fase, em vez de split(), usamos o word_tokenize que sabe que "Harry!" são duas coisas: a palavra "Harry" e o símbolo "!". Depois, filtramos o que não interessa.

In [ ]:
def processamento_top(ficheiros):
    sentences = []
    # 1. Stopwords padrão do NLTK
    stop_words = set(stopwords.words('portuguese')) 
    
    # 2. Stopwords CUSTOMIZADAS (Palavras que aparecem muito mas "sujam" as analogias)
    # Adicionamos verbos comuns e pronomes que o NLTK às vezes deixa passar
    extra_stops = {'disse', 'havia', 'então', 'sobre', 'ainda', 'ia', 'tão', 'ser', 'ter', 'fazer', 'logo', 'vai', 'agora', 'vou'}
    stop_words.update(extra_stops)
    
    # 3. Tabela de tradução para limpar pontuação que o word_tokenize às vezes mantém colada
    translator = str.maketrans('', '', string.punctuation + '“”«»—')

    for ficheiro in ficheiros:
        try:
            with open(ficheiro, 'r', encoding='utf-8') as f:
                texto = f.read().lower()
                
                # Remove pontuação ANTES de tokenizar para evitar "magia."
                texto_limpo = texto.translate(translator)
                
                todas_palavras = word_tokenize(texto_limpo)
                
                # Limpeza final: remove stopwords e palavras muito curtas (menos de 3 letras)
                # Isso remove "ai", "lá", "no", "um" que costumam sujar o gráfico
                frase_limpa = [w for w in todas_palavras if w not in stop_words and len(w) > 2]
                
                # Agrupar em janelas de contexto (n=15 a 20 é o ideal)
                n = 20 
                for i in range(0, len(frase_limpa), n):
                    bloco = frase_limpa[i:i + n]
                    if len(bloco) > 5: # Evita blocos quase vazios no fim do ficheiro
                        sentences.append(bloco)
                    
            print(f"Sucesso ao processar: {ficheiro}")
        except FileNotFoundError:
            print(f"Erro: {ficheiro} não encontrado.")
            
    return sentences

dados_limpos = processamento_top(["harryCamara.txt", "harryPedra.txt"])

Sucesso ao processar: harryCamara.txt
Sucesso ao processar: harryPedra.txt


In [42]:
def realizar_analise_final(model):
    print("=== ANÁLISE DE MÉTRICAS DO MODELO ===\n")

    # 1. TESTE DE SIMILARIDADE (Similarity)
    # Mostra que o modelo entende o contexto de objetos mágicos
    print("1. TESTE DE SIMILARIDADE ALTA:")
    alvo_sim = 'vassoura'
    if alvo_sim in model.wv:
        similares = model.wv.most_similar(alvo_sim, topn=3)
        for s in similares:
            print(f"   {alvo_sim.capitalize()} -> {s[0]}: {s[1]:.4f}")
    else:
        print(f"   Palavra '{alvo_sim}' não encontrada.")

    # 2. TESTE DE ANALOGIA DE LOCALIZAÇÃO (Substituindo Privet por Alfeneiros)
    print("\n2. TESTE DE ANALOGIA (Onde vivem):")
    try:
        # Duda vive em Alfeneiros. Quem vive em Hogwarts?
        res = model.wv.most_similar(positive=['duda', 'hogwarts'], negative=['alfeneiros'], topn=1)
        print(f"   Duda/Alfeneiros -> ?/Hogwarts : {res[0][0]}")
    except KeyError as e:
        print(f"   Erro de chave: {e} (Esta palavra não existe no modelo)")

    # 3. TESTE DE "DOESN'T MATCH" (O Intruso)
    # Nota: Usei 'rony' com 'y' porque é como está no teu meta.tsv
    print("\n3. TESTE DE INTRUSO (Doesnt Match):")
    grupo = ['harry', 'rony', 'hermione', 'voldemort']
    # Verificamos se todas as palavras do grupo existem para não dar erro
    grupo_validado = [w for w in grupo if w in model.wv]
    if len(grupo_validado) > 2:
        intruso = model.wv.doesnt_match(grupo_validado)
        print(f"   No grupo {grupo_validado}, o intruso é: {intruso}")

    # 4. TESTE DE DISTÂNCIA SEMÂNTICA (No Similarity)
    print("\n4. TESTE DE DISTÂNCIA (Mundos Opostos):")
    # Comparar algo muito mágico com algo muito "trouxa"
    if 'magia' in model.wv and 'duda' in model.wv:
        dist = model.wv.similarity('magia', 'duda')
        print(f"   Similaridade entre 'magia' e 'duda': {dist:.4f} (Quanto mais perto de 0, mais longe estão)")

realizar_analise_final(model)

=== ANÁLISE DE MÉTRICAS DO MODELO ===

1. TESTE DE SIMILARIDADE ALTA:
   Vassoura -> ronan: 0.9915
   Vassoura -> certa: 0.9914
   Vassoura -> jogar: 0.9912

2. TESTE DE ANALOGIA (Onde vivem):
   Duda/Alfeneiros -> ?/Hogwarts : vou

3. TESTE DE INTRUSO (Doesnt Match):
   No grupo ['harry', 'rony', 'hermione', 'voldemort'], o intruso é: voldemort

4. TESTE DE DISTÂNCIA (Mundos Opostos):
   Similaridade entre 'magia' e 'duda': 0.7948 (Quanto mais perto de 0, mais longe estão)


In [43]:
try:
    resultado = model.wv.most_similar(positive=['snape', 'grifinória'], negative=['sonserina'], topn=3)
    
    print(f"Snape : Sonserina :: ? : Grifinória")
    for i, res in enumerate(resultado):
        print(f"{i+1}º palpite: {res[0]} ({res[1]:.2f})")

    # Harry está para Grifinória como Draco está para...
    resultado2 = model.wv.most_similar(positive=['draco', 'grifinória'], negative=['sonserina'], topn=1)
    print(f"\nHarry : Grifinória :: Draco : {resultado2[0][0]}")

except Exception as e:
    print(f"Ocorreu um erro: {e}")
    print("Dica: Verifica se as palavras têm acentos no teu ficheiro original.")

print("--- TESTE 1: Identidade ---")
# Harry deve estar muito perto de Potter
print("Próximos de 'harry':", model.wv.most_similar('harry', topn=3))
# Draco deve estar muito perto de Malfoy
print("Próximos de 'draco' :", model.wv.most_similar('draco', topn=3))

Snape : Sonserina :: ? : Grifinória
1º palpite: saiu (0.89)
2º palpite: correndo (0.88)
3º palpite: flitwick (0.88)

Harry : Grifinória :: Draco : hermione
--- TESTE 1: Identidade ---
Próximos de 'harry': [('fazendo', 0.9352321624755859), ('draco', 0.9323699474334717), ('riddle', 0.9303103089332581)]
Próximos de 'draco' : [('fazendo', 0.976852536201477), ('murta', 0.9721238017082214), ('rúbeo', 0.9691099524497986)]


Aqui geramos os ficheiros para carregar no TensorFlow Projector.

In [44]:
# Treinar o modelo
model = Word2Vec(sentences=dados_limpos, vector_size=100, window=5, min_count=2, sg=1)

# Gerar ficheiros TSV para o Projector
out_v = io.open('vecs.tsv', 'w', encoding='utf-8')
out_m = io.open('meta.tsv', 'w', encoding='utf-8')

for word in model.wv.index_to_key:
    vec = model.wv[word]
    out_m.write(word + "\n")
    out_v.write('\t'.join([str(x) for x in vec]) + "\n")
    

out_v.close()
out_m.close()

print("Ficheiros 'vecs.tsv' e 'meta.tsv' prontos para o Projector!")

2026-04-09 15:08:33,430 : INFO : collecting all words and their counts
2026-04-09 15:08:33,430 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2026-04-09 15:08:33,442 : INFO : collected 13258 word types from a corpus of 104073 raw words and 5204 sentences
2026-04-09 15:08:33,443 : INFO : Creating a fresh vocabulary
2026-04-09 15:08:33,450 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=2 retains 6830 unique words (51.52% of original 13258, drops 6428)', 'datetime': '2026-04-09T15:08:33.450740', 'gensim': '4.4.0', 'python': '3.13.7 | packaged by Anaconda, Inc. | (main, Sep  9 2025, 19:54:17) [Clang 17.0.6 ]', 'platform': 'macOS-15.7.4-arm64-arm-64bit-Mach-O', 'event': 'prepare_vocab'}
2026-04-09 15:08:33,451 : INFO : Word2Vec lifecycle event {'msg': 'effective_min_count=2 leaves 97645 word corpus (93.82% of original 104073, drops 6428)', 'datetime': '2026-04-09T15:08:33.451292', 'gensim': '4.4.0', 'python': '3.13.7 | packaged by Anaconda, Inc. |

Ficheiros 'vecs.tsv' e 'meta.tsv' prontos para o Projector!
